In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

INTERNSHIP_DIR = Path("/home/siddiqun/Bureau/Internship")

# Parent directory containing the conformal_envelopes package.
LIBRARY_PARENT = INTERNSHIP_DIR / "Conformal Library"
if str(LIBRARY_PARENT) not in sys.path:
    sys.path.insert(0, str(LIBRARY_PARENT))

from conformal_envelopes import (ConformalSetModel, evaluate_prediction_sets,)

DATA_DIR = INTERNSHIP_DIR / "Data"
SCORES_PATH = DATA_DIR / "phage_host_scores.parquet"
METADATA_PATH = DATA_DIR / "metadata.csv"

GRAM_PRED_PATH = (INTERNSHIP_DIR / "Codes" / "gram_type_predictions.csv")

ALPHA = 0.10
SEED = 13

print("Library imported successfully.")

for path in [SCORES_PATH, METADATA_PATH, GRAM_PRED_PATH]:
    print(f"{path} — exists: {path.is_file()}")
    assert path.is_file(), f"File not found: {path}"

Library imported successfully.
/home/siddiqun/Bureau/Internship/Data/phage_host_scores.parquet — exists: True
/home/siddiqun/Bureau/Internship/Data/metadata.csv — exists: True
/home/siddiqun/Bureau/Internship/Codes/gram_type_predictions.csv — exists: True


## Load scores and metadata

In [2]:
phage_scores = pd.read_parquet(SCORES_PATH)
meta_df = pd.read_csv(METADATA_PATH, index_col="proteinID")

print("Scores shape:", phage_scores.shape)
print("Scores index name:", phage_scores.index.name)
print("Scores index is unique:", phage_scores.index.is_unique)
display(phage_scores.head(3))

print("\nMetadata shape:", meta_df.shape)
print("Metadata columns:", meta_df.columns.tolist())
display(meta_df.head(3))

Scores shape: (18443, 2036)
Scores index name: accession
Scores index is unique: True


,Campylobacter_DNA-associated,Campylobacter_DNA_polymerase,Campylobacter_RNA-associated,Campylobacter_adsorption-related,Campylobacter_annealing,Campylobacter_anti-restriction,Campylobacter_baseplate,Campylobacter_capsid,Campylobacter_cell_wall_depolymerase,Campylobacter_ejection,...,Yersinia_tail,Yersinia_tail_appendage,Yersinia_tail_sheath,Yersinia_terminase,Yersinia_toxin,Yersinia_transcriptional_activator,Yersinia_transcriptional_regulator,Yersinia_transcriptional_repressor,Yersinia_transferase,Yersinia_val
accession,,,,,,,,,,,,,,,,,,,,,
AB002632,0.009630,NaN,NaN,NaN,NaN,NaN,NaN,0.010451,NaN,NaN,...,NaN,NaN,NaN,NaN,0.009864,NaN,0.001164,0.001429,NaN,NaN
AB008550,0.000945,NaN,NaN,0.000533,NaN,0.000509,0.001716,0.111331,0.000244,NaN,...,0.201554,0.22153,0.335059,0.348714,0.006572,0.097914,0.122020,NaN,0.013707,NaN
AB009866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Metadata shape: (1853074, 8)
Metadata columns: ['pc', 'accession', 'vc', 'host', 'host_type', 'phrogs_annotation', 'phrogs_category', 'split']


,pc,accession,vc,host,host_type,phrogs_annotation,phrogs_category,split
proteinID,,,,,,,,
AB002632_00001,KP972568_00002,AB002632,VC_0_0,Vibrio,gram-neg,nicking at origin of replication,"DNA, RNA and nucleotide metabolism",0
AB002632_00002,OP297622_00003,AB002632,VC_0_0,Vibrio,gram-neg,single strand DNA binding protein,"DNA, RNA and nucleotide metabolism",0
AB002632_00003,KC357596_00003,AB002632,VC_0_0,Vibrio,gram-neg,minor coat protein,head and packaging,0


#### Joining metadata at phage level

In [3]:
# Collapse protein metadata to one record per phage.
metadata_columns = ["host", "host_type", "split"]
grouped = meta_df.groupby("accession")[metadata_columns]

# Avoid silently selecting conflicting metadata.
conflicts = grouped.nunique().gt(1).any(axis=1)
assert not conflicts.any(), (f"Conflicting metadata for: " f"{conflicts[conflicts].index[:10].tolist()}")

phage_meta = grouped.first()

score_cols = phage_scores.columns.tolist()
all_hosts = sorted({col.split("_", 1)[0] for col in score_cols})

phage_df = phage_scores.join(phage_meta, how="inner", validate="one_to_one",)

assert phage_df[["host", "split"]].notna().all().all()

# Keep hosts represented by the predictor columns.
phage_df = phage_df.loc[phage_df["host"].isin(all_hosts)].copy()
phage_df["split"] = phage_df["split"].astype(int)
phage_df = phage_df.rename_axis("ID").reset_index()

# Preserve the original partition.
train = phage_df.loc[~phage_df["split"].isin([0, 1])].copy()
validation = phage_df.loc[phage_df["split"].eq(1)].copy()
test = phage_df.loc[phage_df["split"].eq(0)].copy()

print("Training phages:", len(train))
print("Validation phages:", len(validation))
print("Test phages:", len(test))
print("Candidate hosts:", len(all_hosts))

Training phages: 9101
Validation phages: 2593
Test phages: 3944
Candidate hosts: 54


## Define each host’s score columns

In [4]:
train_counts = train["host"].value_counts()

fitted_hosts = [host for host in all_hosts if train_counts.get(host, 0) >= 3]

skipped_hosts = sorted(set(all_hosts) - set(fitted_hosts))

label_to_columns = {host: [col for col in score_cols if col.split("_", 1)[0] == host]
                    for host in fitted_hosts}

fit_train = train.loc[train["host"].isin(fitted_hosts)].copy()

# Check that every training phage has a score for its true host.
missing_true_host = 0

for host, columns in label_to_columns.items():
    assert columns, f"No score columns for {host}"

    host_scores = fit_train.loc[fit_train["host"].eq(host), columns]
    missing_true_host += int(host_scores.isna().all(axis=1).sum())

print("Hosts eligible for fitting:", len(fitted_hosts))
print("Skipped hosts:", skipped_hosts)
print("Training rows retained:", len(fit_train))
print("Training phages without true-host scores:", missing_true_host)

assert missing_true_host == 0, "Some training rows lack true-host scores."

Hosts eligible for fitting: 54
Skipped hosts: []
Training rows retained: 9101
Training phages without true-host scores: 0


### Loading the Gram predictions

In [5]:
gram_pred_df = pd.read_csv(GRAM_PRED_PATH).set_index("accession")

assert gram_pred_df.index.is_unique, "Duplicate accession IDs."

# Convert CSV values explicitly to booleans.
for column in ["pred_gramneg", "pred_grampos"]:
    gram_pred_df[column] = (gram_pred_df[column].astype(str).str.strip().str.lower()
                            .map({"true": True, "false": False, "1": True, "0": False,}))

    assert gram_pred_df[column].notna().all(), (f"Invalid or missing values in {column}")

validation_covered = validation["ID"].isin(gram_pred_df.index)

print("Gram prediction file:", GRAM_PRED_PATH)
print("Phages in export:", len(gram_pred_df))
print("Validation IDs covered:", int(validation_covered.sum()))

display(gram_pred_df[["pred_gramneg", "pred_grampos"]].value_counts().rename("phages"))

assert validation_covered.all(), ("Some validation IDs are missing from the Gram prediction file.")

Gram prediction file: /home/siddiqun/Bureau/Internship/Codes/gram_type_predictions.csv
Phages in export: 18474
Validation IDs covered: 2593


pred_gramneg  pred_grampos
True          False           10744
False         True             7730
Name: phages, dtype: int64

#### Constructing candidates from the predicted Gram types and checking whether those candidates have usable scores.

In [6]:
# Map candidate hosts to their Gram group.
host_types = (phage_df[["host", "host_type"]].dropna().drop_duplicates())

assert host_types.groupby("host")["host_type"].nunique().le(1).all()

host_to_gram = host_types.set_index("host")["host_type"].to_dict()

assert all(host_to_gram.get(host) in {"gram-neg", "gram-pos"} for host in fitted_hosts)

# Select candidates using predicted Gram types.
validation_candidates = {}

for sample_id in validation["ID"]:
    prediction = gram_pred_df.loc[sample_id]

    allowed = [host for host in fitted_hosts if (host_to_gram[host] == "gram-neg"
                                                 and prediction["pred_gramneg"]) 
               or (host_to_gram[host] == "gram-pos" and prediction["pred_grampos"])]

    # Match the original notebook's fallback.
    validation_candidates[sample_id] = allowed or fitted_hosts.copy()

# Check availability using columns the collapsed model would retain.
missing_pairs = 0
affected_ids = set()

for host, columns in label_to_columns.items():
    training_scores = fit_train.loc[fit_train["host"].eq(host), columns]
    used_columns = training_scores.columns[training_scores.notna().any(axis=0)].tolist()

    allowed_mask = validation["ID"].map(lambda sample_id: host in validation_candidates[sample_id])
    missing_mask = validation[used_columns].isna().all(axis=1)
    problematic = allowed_mask & missing_mask

    missing_pairs += int(problematic.sum())
    affected_ids.update(validation.loc[problematic, "ID"])

print("Candidate lists created:", len(validation_candidates))
print("Allowed phage–host pairs without scores:", missing_pairs)
print("Validation phages affected:", len(affected_ids))

Candidate lists created: 2593
Allowed phage–host pairs without scores: 32
Validation phages affected: 2


## Collapsed Model

In [7]:
collapsed_model = ConformalSetModel(method="collapsed", alpha=ALPHA, score_direction="higher_is_better",
                                    shape_fraction=0.5, random_state=SEED, force_nonempty=False,)

collapsed_model.fit(fit_train, id_col="ID", label_col="host", score_cols=score_cols, 
                    label_to_columns=label_to_columns,)

envelopes = collapsed_model.get_envelopes()

print("Collapsed envelopes fitted:", len(envelopes))
print("Infinite calibration thresholds:", sum(np.isposinf(info["envelope"]["t_hat"])
                                              for info in envelopes.values()),)

Collapsed envelopes fitted: 54
Infinite calibration thresholds: 12


#### Predicting on validation data

In [8]:
collapsed_predictions = collapsed_model.predict_per_sample(validation, candidates_by_id=validation_candidates,)

# Align true labels with prediction IDs.
true_labels = (validation.set_index("ID")["host"].loc[collapsed_predictions["ID"]].tolist())

collapsed_metrics = evaluate_prediction_sets(collapsed_predictions["prediction_set"], true_labels,)

print("Validation predictions:", len(collapsed_predictions))
print(f"Coverage: {collapsed_metrics['coverage']:.3f}")
print(f"Average set size: {collapsed_metrics['average_set_size']:.2f}")
print(f"Singleton rate: {collapsed_metrics['singleton_rate']:.3f}")
print(f"Empty-set rate: {collapsed_metrics['empty_rate']:.3f}")
print("Forced predictions:", int(collapsed_predictions["forced"].sum()))

display(collapsed_predictions[["ID", "prediction_set", "set_size", "forced"]].head())

Validation predictions: 2593
Coverage: 0.870
Average set size: 10.69
Singleton rate: 0.000
Empty-set rate: 0.001
Forced predictions: 0


,ID,prediction_set,set_size,forced
0,AB008550,"[Citrobacter, Cronobacter, Dickeya, Edwardsiel...",18,False
1,AB009866,"[Clostridioides, Lacticaseibacillus, Staphyloc...",3,False
2,AB044554,"[Clostridioides, Lacticaseibacillus, Staphyloc...",3,False
3,AB243556,"[Clostridioides, Lacticaseibacillus, Staphyloc...",3,False
4,AB362338,"[Cronobacter, Dickeya, Edwardsiella, Enterobac...",18,False


In [9]:
diagnostics = collapsed_predictions[["ID", "set_size"]].copy()

diagnostics["true_host"] = true_labels

diagnostics["true_host_allowed"] = [host in validation_candidates[sample_id]
                                    for sample_id, host in zip(diagnostics["ID"], 
                                                               diagnostics["true_host"])]

diagnostics["covered"] = [host in prediction_set for host, 
                          prediction_set in zip(diagnostics["true_host"], 
                                                collapsed_predictions["prediction_set"],)]

excluded = ~diagnostics["true_host_allowed"]
rejected = diagnostics["true_host_allowed"] & ~diagnostics["covered"]

print("True host excluded by Gram filtering:", int(excluded.sum()))
print("True host allowed but rejected by envelope:", int(rejected.sum()))
print("Total coverage misses:", int((~diagnostics["covered"]).sum()))
print("Empty prediction sets:", int(diagnostics["set_size"].eq(0).sum()))

per_host = (diagnostics.groupby("true_host").agg(validation_phages=("covered", "size"),
                                                 coverage=("covered", "mean"), 
                                                 gram_retention=("true_host_allowed", "mean"),
                                                 average_set_size=("set_size", "mean"),)
                                                 .sort_values(["coverage", "validation_phages"], 
                                                              ascending=[True, False]))

display(per_host)

True host excluded by Gram filtering: 2
True host allowed but rejected by envelope: 336
Total coverage misses: 338
Empty prediction sets: 2


,validation_phages,coverage,gram_retention,average_set_size
true_host,,,,
Caulobacter,1,0.000000,1.000000,18.000000
Flavobacterium,74,0.162162,1.000000,12.648649
Yersinia,41,0.195122,1.000000,19.560976
Cellulophaga,23,0.391304,1.000000,14.347826
Klebsiella,103,0.446602,1.000000,18.300971
Paenibacillus,6,0.500000,1.000000,2.500000
Prochlorococcus,10,0.700000,1.000000,15.100000
Stenotrophomonas,7,0.714286,0.857143,14.714286
Gordonia,81,0.728395,1.000000,3.049383


In [10]:
checks = []

for host in ["Flavobacterium", "Yersinia", "Cellulophaga", "Klebsiella"]:
    info = collapsed_model.get_envelopes()[host]
    envelope = info["envelope"]
    columns = info["columns"]

    training_raw = fit_train.loc[fit_train["host"].eq(host), columns].to_numpy(dtype=float)

    calibration_raw = training_raw[info["idx_s2"]]
    calibration_scores = np.nanmean(1.0 - calibration_raw, axis=1)

    n = len(calibration_scores)
    k = int(np.ceil((n + 1) * (1 - ALPHA)))

    direct_boundary = (np.sort(calibration_scores)[k - 1] if k <= n else np.inf)

    library_boundary = ((envelope["q_tilde"] + 1e-12) * envelope["t_hat"])

    validation_raw = validation.loc[validation["host"].eq(host), columns].to_numpy(dtype=float)

    validation_scores = np.nanmean(1.0 - validation_raw, axis=1)

    checks.append({"host": host, "calibration_n": n, "direct_boundary": direct_boundary, "library_boundary": library_boundary,
                   "boundaries_match": np.isclose(direct_boundary, library_boundary),
                   "calibration_acceptance": np.mean(calibration_scores <= direct_boundary),
                   "validation_acceptance": np.mean(validation_scores <= direct_boundary),
                   "calibration_median": np.median(calibration_scores),
                   "validation_median": np.median(validation_scores),})

display(pd.DataFrame(checks))

,host,calibration_n,direct_boundary,library_boundary,boundaries_match,calibration_acceptance,validation_acceptance,calibration_median,validation_median
0,Flavobacterium,73,0.744849,0.744849,True,0.931507,0.162162,0.680336,0.811577
1,Yersinia,34,0.913603,0.913603,True,0.941176,0.195122,0.859282,0.948051
2,Cellulophaga,11,0.756058,0.756058,True,1.000000,0.391304,0.734540,0.809175
3,Klebsiella,348,0.909848,0.909848,True,0.905172,0.446602,0.847585,0.919585


## Radial Envelope

In [ ]:
radial_model = ConformalSetModel(method="radial", alpha=ALPHA, score_direction="higher_is_better", 
                                 shape_fraction=0.5, random_state=SEED, force_nonempty=False, 
                                 n_directions=100, smoothing=4.0, angle_deg=30.0, neighbor_fraction=0.2,)

radial_model.fit(fit_train, id_col="ID", label_col="host", score_cols=score_cols, label_to_columns=label_to_columns,)

radial_envelopes = radial_model.get_envelopes()

print("Radial envelopes fitted:", len(radial_envelopes))
print("Infinite calibration thresholds:", sum(np.isposinf(info["envelope"]["t_hat"])
                                              for info in radial_envelopes.values()),)

In [8]:
# Inspect two representative hosts from opposite Gram groups.
for candidate_host in ["Escherichia", "Staphylococcus"]:
    columns = collapsed_model.get_envelopes()[candidate_host]["columns"]
    has_scores = validation[columns].notna().any(axis=1)

    print(f"\nScores available for candidate host: {candidate_host}")
    display(pd.crosstab(
        validation["host_type"].fillna("unknown"),
        has_scores,
        rownames=["True Gram type"],
        colnames=["Has candidate-host scores"],
        dropna=False,
    ))

# Check whether any validation phage lacks scores for its true host.
true_host_missing = []

for host, info in collapsed_model.get_envelopes().items():
    rows = validation.loc[validation["host"].eq(host)]
    missing = rows[info["columns"]].isna().all(axis=1)
    true_host_missing.extend(rows.loc[missing, "ID"].tolist())

print(
    "\nValidation phages without true-host scores:",
    len(true_host_missing),
)
print("Example IDs:", true_host_missing[:10])


Scores available for candidate host: Escherichia


Has candidate-host scores,False,True
True Gram type,,
gram-neg,0,1397
gram-pos,1196,0



Scores available for candidate host: Staphylococcus


Has candidate-host scores,False,True
True Gram type,,
gram-neg,1397,0
gram-pos,0,1196



Validation phages without true-host scores: 0
Example IDs: []


In [ ]:
# The Gram notebook saves this file in its working directory.
internship_dir = DATA_DIR.resolve().parent

gram_files = sorted(internship_dir.rglob("gram_type_predictions.csv"))

print("Matching Gram prediction files:")
for path in gram_files:
    print(path)

if len(gram_files) == 1:
    GRAM_PRED_PATH = gram_files[0]
    print("\nSelected:", GRAM_PRED_PATH)
elif not gram_files:
    print("\nNo file found. Check where the Gram notebook saved its export.")
else:
    print("\nMultiple files found. We need to select the intended export.")

Matching Gram prediction files:
/home/siddiqun/Bureau/Internship/Codes/Gram Type Classification/gram_type_predictions.csv
/home/siddiqun/Bureau/Internship/Codes/gram_type_predictions.csv

Multiple files found. We need to select the intended export.


In [10]:
gram_exports = {}

for path in gram_files:
    df = pd.read_csv(path)

    required = ["accession", "pred_gramneg", "pred_grampos"]
    assert set(required).issubset(df.columns), f"Missing columns: {path}"
    assert df["accession"].is_unique, f"Duplicate accessions: {path}"

    df = df[required].set_index("accession").sort_index()

    for column in ["pred_gramneg", "pred_grampos"]:
        df[column] = (
            df[column].astype(str).str.strip().str.lower()
            .map({"true": True, "false": False, "1": True, "0": False})
        )
        assert df[column].notna().all(), f"Invalid values: {path}"

    gram_exports[path] = df

    print(f"\n{path}")
    print("Phages:", len(df))
    print("Validation IDs covered:", validation["ID"].isin(df.index).sum())
    display(df.value_counts().rename("phages"))

first, second = gram_files
print(
    "\nIdentical prediction tables:",
    gram_exports[first].equals(gram_exports[second]),
)


/home/siddiqun/Bureau/Internship/Codes/Gram Type Classification/gram_type_predictions.csv
Phages: 18474
Validation IDs covered: 2593


pred_gramneg  pred_grampos
True          False           10241
False         True             6826
True          True             1407
Name: phages, dtype: int64


/home/siddiqun/Bureau/Internship/Codes/gram_type_predictions.csv
Phages: 18474
Validation IDs covered: 2593


pred_gramneg  pred_grampos
True          False           10744
False         True             7730
Name: phages, dtype: int64


Identical prediction tables: False


In [10]:
GRAM_PRED_PATH = (
    DATA_DIR.resolve().parent / "Codes" / "gram_type_predictions.csv"
)
gram_pred_df = gram_exports[GRAM_PRED_PATH]

host_to_gram = (
    phage_df[["host", "host_type"]]
    .dropna()
    .drop_duplicates()
    .set_index("host")["host_type"]
    .to_dict()
)

assert all(
    host_to_gram.get(host) in {"gram-neg", "gram-pos"}
    for host in fitted_hosts
)

validation_candidates = {}

for sample_id in validation["ID"]:
    row = gram_pred_df.loc[sample_id]

    allowed = [
        host for host in fitted_hosts
        if (
            host_to_gram[host] == "gram-neg"
            and row["pred_gramneg"]
        ) or (
            host_to_gram[host] == "gram-pos"
            and row["pred_grampos"]
        )
    ]

    # Match the original notebook's fallback.
    validation_candidates[sample_id] = allowed or fitted_hosts.copy()

missing_allowed = []

for host, info in collapsed_model.get_envelopes().items():
    allowed_mask = validation["ID"].map(
        lambda sample_id: host in validation_candidates[sample_id]
    )
    missing_mask = validation[info["columns"]].isna().all(axis=1)

    for sample_id in validation.loc[allowed_mask & missing_mask, "ID"]:
        missing_allowed.append({
            "ID": sample_id,
            "candidate_host": host,
        })

print("Selected export:", GRAM_PRED_PATH)
print("Allowed phage–host pairs without scores:", len(missing_allowed))
print(
    "Validation phages affected:",
    len({row["ID"] for row in missing_allowed}),
)

display(pd.DataFrame(missing_allowed).head(10))

NameError: name 'gram_exports' is not defined

In [ ]:
method_params = {
    "collapsed": {},
    "radial": {
        "n_directions": 100,
        "smoothing": 4.0,
        "angle_deg": 30.0,
        "neighbor_fraction": 0.2,
    },
    "strip": {
        "n_bins": 10,
        "smoothing_window": 2,
        "min_samples": 3,
    },
}

models = {}

for method, params in method_params.items():
    print(f"Fitting {method}...")

    model = ConformalSetModel(
        method=method,
        alpha=ALPHA,
        shape_fraction=0.5,
        random_state=SEED,
        force_nonempty=False,
        **params,
    )

    model.fit(
        fit_train,
        id_col="ID",
        label_col="host",
        score_cols=score_cols,
        label_to_columns=label_to_columns,
    )

    models[method] = model

print("All models fitted.")

## Load predicted Gram types and construct candidates

In [ ]:
gram_predictions = pd.read_csv(GRAM_PRED_PATH).set_index("accession")
assert gram_predictions.index.is_unique

for column in ["pred_gramneg", "pred_grampos"]:
    gram_predictions[column] = (
        gram_predictions[column]
        .astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )
    assert gram_predictions[column].notna().all(), (
        f"Invalid values in {column}"
    )

host_types = (
    phage_df[["host", "host_type"]]
    .dropna()
    .drop_duplicates()
)

assert host_types.groupby("host")["host_type"].nunique().le(1).all()
host_to_gram = host_types.set_index("host")["host_type"].to_dict()

assert all(
    host_to_gram.get(host) in {"gram-neg", "gram-pos"}
    for host in fitted_hosts
), "Missing or unexpected host Gram-type metadata."


def candidates_for(sample_id):
    if sample_id not in gram_predictions.index:
        return fitted_hosts.copy()

    row = gram_predictions.loc[sample_id]

    allowed = [
        host for host in fitted_hosts
        if (
            host_to_gram[host] == "gram-neg"
            and row["pred_gramneg"]
        ) or (
            host_to_gram[host] == "gram-pos"
            and row["pred_grampos"]
        )
    ]

    return allowed or fitted_hosts.copy()

## Evaluate on validation first

In [ ]:
evaluation = validation

candidates_by_id = {
    sample_id: candidates_for(sample_id)
    for sample_id in evaluation["ID"]
}

truth_by_id = evaluation.set_index("ID")["host"]

predictions_by_method = {}
summary = []

for method, model in models.items():
    for force in [False, True]:
        model.force_nonempty = force

        predictions = model.predict_per_sample(
            evaluation,
            candidates_by_id=candidates_by_id,
        )

        # Align truth explicitly by ID.
        truth = truth_by_id.loc[predictions["ID"]].tolist()

        metrics = evaluate_prediction_sets(
            predictions["prediction_set"],
            truth,
        )

        predictions_by_method[(method, force)] = predictions

        summary.append({
            "method": method,
            "force_nonempty": force,
            "coverage": metrics["coverage"],
            "average_set_size": metrics["average_set_size"],
            "empty_rate": metrics["empty_rate"],
            "singleton_rate": metrics["singleton_rate"],
            "forced_rate": predictions["forced"].mean(),
        })

    model.force_nonempty = False

display(pd.DataFrame(summary))